# This file is for experimenting and seeing what works and how. All the final code will be available in main.py.

In [33]:
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
import os
import pickle

In [34]:
scope = ["https://www.googleapis.com/auth/gmail.modify"]

In [35]:
creds = None

In [36]:
if os.path.exists("token.pickle"):
    with open("token.pickle", "rb") as token:
        creds = pickle.load(token)

In [37]:
if not creds:
    flow = InstalledAppFlow.from_client_secrets_file(
        "credentials.json",
        scope
    )

    creds = flow.run_local_server(port=0)

    with open("token.pickle", "wb") as token:
        pickle.dump(creds, token)

In [38]:
service = build("gmail", "v1", credentials=creds)

In [39]:
from pprint import pprint
results = service.users().messages().list(
    userId="me"
).execute()

pprint(results)

{'messages': [{'id': '19e9150e33bcc680', 'threadId': '19e9150e33bcc680'},
              {'id': '19e8de893a2d7644', 'threadId': '19e8de893a2d7644'},
              {'id': '19e8dac9ddb0ffcd', 'threadId': '19e8dac9ddb0ffcd'},
              {'id': '19e893ca0485ec4e', 'threadId': '19e893ca0485ec4e'},
              {'id': '19e88cb9357ec75a', 'threadId': '19e88cb9357ec75a'},
              {'id': '19e87c437556c129', 'threadId': '19e87c437556c129'},
              {'id': '19e87b8e8a0702d3', 'threadId': '19e8728dd7ebe661'},
              {'id': '19e874ac118e5d60', 'threadId': '19e8728dd7ebe661'},
              {'id': '19e8728dd7ebe661', 'threadId': '19e8728dd7ebe661'},
              {'id': '19e86e93c3f2eee7', 'threadId': '19e86e93c3f2eee7'},
              {'id': '19e86e3785aca4e2', 'threadId': '19e86e3785aca4e2'},
              {'id': '19e86e17407f2592', 'threadId': '19e86e17407f2592'},
              {'id': '19e86e1548eecba3', 'threadId': '19e86e1548eecba3'},
              {'id': '19e85484437ef053

In [40]:
profile = service.users().getProfile(userId="me").execute()

print(profile)

{'emailAddress': 'syedmrihaan@gmail.com', 'messagesTotal': 1198, 'threadsTotal': 1103, 'historyId': '597576'}


In [41]:
results = service.users().messages().list(
    userId="me",
    labelIds=["INBOX"]
).execute()

print(results["resultSizeEstimate"])

201


# **Our OAuth has been implemented.**

### **Next Steps:**
***Data Collection***
- Collect message ID
- Collect sender name
- Collect sender addresses
- Collect combined text(subject + body)
- Collect word count of body
- Collect number of numerical characters(digits) in body.
- Collect Headers
- Collect mail folder

***Final Dataframe***
- Sender info 
- Combined text
- Word count
- Digit count
- Mail folder
- Message ID(DO NOT TRAIN ON THIS)

***Marking Targets***
- Perform unsupervised KMeans to get clusters.

***Model and Final pipeline***
- Gaussian naive bayes.

In [47]:
# get msg IDs which we will work with

ids = service.users().messages().list(
    userId = "me",
    q = "newer_than:90d",
    maxResults = 500
).execute()

In [48]:
ids

{'messages': [{'id': '19e9150e33bcc680', 'threadId': '19e9150e33bcc680'},
  {'id': '19e8de893a2d7644', 'threadId': '19e8de893a2d7644'},
  {'id': '19e8dac9ddb0ffcd', 'threadId': '19e8dac9ddb0ffcd'},
  {'id': '19e893ca0485ec4e', 'threadId': '19e893ca0485ec4e'},
  {'id': '19e88cb9357ec75a', 'threadId': '19e88cb9357ec75a'},
  {'id': '19e87c437556c129', 'threadId': '19e87c437556c129'},
  {'id': '19e87b8e8a0702d3', 'threadId': '19e8728dd7ebe661'},
  {'id': '19e874ac118e5d60', 'threadId': '19e8728dd7ebe661'},
  {'id': '19e8728dd7ebe661', 'threadId': '19e8728dd7ebe661'},
  {'id': '19e86e93c3f2eee7', 'threadId': '19e86e93c3f2eee7'},
  {'id': '19e86e3785aca4e2', 'threadId': '19e86e3785aca4e2'},
  {'id': '19e86e17407f2592', 'threadId': '19e86e17407f2592'},
  {'id': '19e86e1548eecba3', 'threadId': '19e86e1548eecba3'},
  {'id': '19e85484437ef053', 'threadId': '19e85484437ef053'},
  {'id': '19e82e2fa5108139', 'threadId': '19e82e2fa5108139'},
  {'id': '19e81f0ad8549371', 'threadId': '19e81f0ad8549371

In [49]:
ids = ids["messages"]

In [50]:
ids

[{'id': '19e9150e33bcc680', 'threadId': '19e9150e33bcc680'},
 {'id': '19e8de893a2d7644', 'threadId': '19e8de893a2d7644'},
 {'id': '19e8dac9ddb0ffcd', 'threadId': '19e8dac9ddb0ffcd'},
 {'id': '19e893ca0485ec4e', 'threadId': '19e893ca0485ec4e'},
 {'id': '19e88cb9357ec75a', 'threadId': '19e88cb9357ec75a'},
 {'id': '19e87c437556c129', 'threadId': '19e87c437556c129'},
 {'id': '19e87b8e8a0702d3', 'threadId': '19e8728dd7ebe661'},
 {'id': '19e874ac118e5d60', 'threadId': '19e8728dd7ebe661'},
 {'id': '19e8728dd7ebe661', 'threadId': '19e8728dd7ebe661'},
 {'id': '19e86e93c3f2eee7', 'threadId': '19e86e93c3f2eee7'},
 {'id': '19e86e3785aca4e2', 'threadId': '19e86e3785aca4e2'},
 {'id': '19e86e17407f2592', 'threadId': '19e86e17407f2592'},
 {'id': '19e86e1548eecba3', 'threadId': '19e86e1548eecba3'},
 {'id': '19e85484437ef053', 'threadId': '19e85484437ef053'},
 {'id': '19e82e2fa5108139', 'threadId': '19e82e2fa5108139'},
 {'id': '19e81f0ad8549371', 'threadId': '19e81f0ad8549371'},
 {'id': '19e7e88c44915c1

In [51]:
len(ids)

273

In [64]:
m_id = ids[5]["id"]

In [65]:
msg = service.users().messages().get(
    userId = "me",
    id = m_id
).execute()

In [ ]:
msg # plug into json crack to visualize data

{'id': '19e87c437556c129',
 'threadId': '19e87c437556c129',
 'labelIds': ['INBOX'],
 'snippet': 'Get back to listening with friends. Get Spotify Premium. Restart the party with 2 months of Premium Standard for ₹99 Come back to Spotify Premium Standard to play your favorite albums and playlists',
 'payload': {'partId': '',
  'mimeType': 'multipart/alternative',
  'filename': '',
  'headers': [{'name': 'Delivered-To', 'value': 'syedmrihaan@gmail.com'},
   {'name': 'Received',
    'value': 'by 2002:a05:6236:5985:b0:1d8:f5e6:d9e7 with SMTP id bt5csp2246324ggc;        Tue, 2 Jun 2026 02:57:23 -0700 (PDT)'},
   {'name': 'X-Received',
    'value': 'by 2002:a05:622a:102:b0:516:ccc0:ee40 with SMTP id d75a77b69052e-5175bef7dc4mr82237441cf.57.1780394243784;        Tue, 02 Jun 2026 02:57:23 -0700 (PDT)'},
   {'name': 'ARC-Seal',
    'value': 'i=1; a=rsa-sha256; t=1780394243; cv=none;        d=google.com; s=arc-20240605;        b=NjgnLAGzyO/tQa4GVcaIR1lDiQ3dv2aJ7pn+MYImVVJBW9Z1kghsvIhUXpJvuqsuBZ   

In [ ]:
def get_header(msg, name):
    for h in msg["payload"]["headers"]:
        if h["name"] == name:
            return h["value"]
    return None

In [68]:
records = []
for m_id in ids:
    m_id = m_id["id"]
    record = []
    msg = service.users().messages().get(
    userId = "me",
    id = m_id
    ).execute()
    subject = get_header(msg, "Subject")
    record.append(subject)#add subject
    records.append(record)# append record to records

In [69]:
records

[['[GitHub] A third-party GitHub Application has been added to your account'],
 ['100 words every adult should know'],
 ['Hi Muhammad, start sharing today!'],
 ['Action Required: Important security update for OpenAI macOS apps'],
 [' Invitation | Thinking Like Cambridge: Quantum Computing and Academic Excellence Masterclass'],
 ['Bring the band back together with Spotify Premium for ₹99'],
 ['Fwd: Visa Invitation Letter for AIYA S26 Programme'],
 ['Fwd: Visa Invitation Letter for AIYA S26 Programme'],
 ['Visa Invitation Letter for AIYA S26 Programme'],
 ['Welcome to Hackatime!'],
 ['[house-prices…] Comment: '],
 ['695-913 is your Hack Club login code'],
 ['Welcome to Hack Club Stardance ^_^'],
 ['Hack Club 🤝 NASA - Stardance Challenge'],
 ['New Model: Claude Opus 4.8 now live in Julius'],
 ['[Rainbow Vistas @ Rock Garden] Maintenance Charge raised'],
 ['Security alert'],
 ['Security alert'],
 ['Security alert'],
 ['Security alert'],
 ['Security alert'],
 ['Security alert'],
 ['Security